# Dublagem — beta

Pega um áudio em português e devolve o mesmo conteúdo em outro idioma, **na voz da pessoa**.

Três etapas: **transcreve** (Whisper) → **adapta** (Gemini) → **sintetiza clonando a voz** (XTTS v2).

---

### Antes de rodar

**1. Liga a GPU.** Menu `Ambiente de execução` → `Alterar o tipo de ambiente` → **T4 GPU**. Sem GPU o passo 4 leva minutos por frase e não vale a pena.

**2. Licença.** O modelo XTTS v2 é **não comercial** (CPML). Isto aqui é teste de avaliação. Antes de usar em anúncio de verdade, troque o motor de voz — o OpenVoice v2 (MIT) é o primeiro a olhar.

**3. Autorização.** Clonar a voz de alguém exige permissão dessa pessoa.

## 1. Instalar (leva 3–5 minutos, roda uma vez por sessão)

In [ ]:
# coqui-tts é o fork mantido do projeto original (a Coqui fechou em 2024).
# faster-whisper transcreve bem mais rápido que o whisper original, mesmo modelo.
!pip install -q coqui-tts faster-whisper pydub google-genai
!apt-get -qq install -y ffmpeg > /dev/null

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — ligue a T4 no menu Ambiente de execução')

## 2. Enviar o áudio

Qualquer formato. O áudio é convertido pra WAV 22 kHz mono, que é o que o clonador espera.

> Qualidade da gravação importa mais que duração. Áudio de WhatsApp (comprimido) clona pior que gravação limpa — se o resultado sair estranho, suspeite da fonte antes da ferramenta.

In [ ]:
from google.colab import files
import subprocess, os

print('Escolha o áudio em português:')
enviado = files.upload()
origem = list(enviado.keys())[0]

ORIGINAL = 'original.wav'
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', origem,
                '-vn', '-ac', '1', '-ar', '22050', ORIGINAL], check=True)

dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                            '-of', 'csv=p=0', ORIGINAL], capture_output=True, text=True).stdout)
print(f'pronto: {ORIGINAL} · {dur:.1f}s')
if dur < 6:
    print('AVISO: menos de 6s de referência costuma gerar clone ruim.')

## 3. Transcrever

In [ ]:
from faster_whisper import WhisperModel

modelo = WhisperModel('large-v3', device='cuda', compute_type='float16')
segmentos, info = modelo.transcribe(ORIGINAL, language='pt', vad_filter=True)

TEXTO_PT = ' '.join(s.text.strip() for s in segmentos).strip()
print(f'{info.duration:.1f}s · {len(TEXTO_PT.split())} palavras\n')
print(TEXTO_PT)

## 4. Adaptar para o idioma alvo (Gemini)

**Não é tradução literal, é adaptação.** Copy de anúncio traduzida ao pé da letra fica correta e morta: gancho, gíria e prova social não atravessam idioma.

A chave do Gemini é a mesma que está salva no The Track (Configurações → IA). Ela é pedida escondida e não fica gravada no notebook.

In [ ]:
from getpass import getpass
from google import genai
from google.genai import types

IDIOMA = 'inglês americano'   # troque aqui: 'espanhol', 'francês'...
CONTEXTO = 'Anúncio de renda extra para mulheres, tom informal de conversa.'

API_KEY = getpass('Cole a chave do Gemini: ')
cliente = genai.Client(api_key=API_KEY)

INSTRUCAO = f'''Você adapta roteiro de anúncio falado, de português para {IDIOMA}.

Isto NÃO é tradução. É reescrita para soar nativa:
- mantenha a mesma intenção, a mesma promessa e a mesma ordem das ideias;
- troque gíria, expressão e referência cultural pelo equivalente natural;
- mantenha o ritmo de fala e o tom de conversa, não de texto escrito;
- mantenha a duração parecida: fala falada, não parágrafo;
- não acrescente nada que o original não diga.

Contexto do anúncio: {CONTEXTO}

Responda SÓ com o texto adaptado, sem aspas e sem comentário.'''

resposta = cliente.models.generate_content(
    model='gemini-3.5-flash',
    contents=TEXTO_PT,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCAO,
        temperature=0.7,
        # Folga grande: os modelos que "pensam" gastam o mesmo orçamento no
        # raciocínio, e um teto apertado devolve resposta vazia.
        max_output_tokens=4000,
    ),
)

TEXTO_ALVO = (resposta.text or '').strip()
if not TEXTO_ALVO:
    raise SystemExit('O Gemini respondeu vazio. Aumente max_output_tokens ou troque pro gemini-3.5-flash.')

print('--- ORIGINAL ---\n' + TEXTO_PT)
print('\n--- ADAPTADO ---\n' + TEXTO_ALVO)
print(f'\npalavras: {len(TEXTO_PT.split())} → {len(TEXTO_ALVO.split())}')

### Ajuste à mão (opcional)

Se a adaptação não ficou boa, corrija o texto aqui antes de gastar GPU sintetizando.

In [ ]:
# Descomente e edite se quiser mandar um texto seu:
# TEXTO_ALVO = """cole aqui a versão que você quer ouvir"""
print(TEXTO_ALVO)

## 5. Sintetizar na voz original

O XTTS tem limite de texto por geração, então a fala é quebrada em frases e emendada no fim — senão o texto inteiro de 50 segundos é cortado no meio sem avisar.

In [ ]:
import os, re, torch
os.environ['COQUI_TOS_AGREED'] = '1'   # aceita a licença do modelo (não comercial)

from TTS.api import TTS
from pydub import AudioSegment

CODIGO = {'inglês americano': 'en', 'inglês': 'en', 'espanhol': 'es',
          'francês': 'fr', 'italiano': 'it', 'alemão': 'de'}[IDIOMA]

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')

def frases(texto, teto=220):
    """Quebra em frases, juntando as curtas até o teto de caracteres."""
    partes, atual = [], ''
    for p in re.split(r'(?<=[.!?])\s+', texto.strip()):
        if len(atual) + len(p) + 1 <= teto:
            atual = (atual + ' ' + p).strip()
        else:
            if atual:
                partes.append(atual)
            atual = p
    if atual:
        partes.append(atual)
    return partes

pedacos = frases(TEXTO_ALVO)
print(f'{len(pedacos)} pedaço(s) para sintetizar')

juntos = AudioSegment.silent(duration=0)
for i, p in enumerate(pedacos, 1):
    print(f'  {i}/{len(pedacos)}: {p[:60]}...')
    tts.tts_to_file(text=p, speaker_wav=ORIGINAL, language=CODIGO, file_path=f'p{i}.wav')
    # 120ms entre frases: sem respiro a fala emendada soa apressada.
    juntos += AudioSegment.from_wav(f'p{i}.wav') + AudioSegment.silent(duration=120)

SAIDA = 'dublado.wav'
juntos.export(SAIDA, format='wav')
print(f'\npronto: {SAIDA} · {len(juntos)/1000:.1f}s (original tinha {dur:.1f}s)')

## 6. Ouvir e baixar

In [ ]:
from IPython.display import Audio, display

print('ORIGINAL:'); display(Audio(ORIGINAL))
print('DUBLADO:');  display(Audio(SAIDA))

In [ ]:
from google.colab import files
files.download(SAIDA)

---

## O que julgar no resultado

Ouça procurando três coisas, nesta ordem:

1. **É a voz dela?** Timbre reconhecível. Se não for, o problema é a gravação de referência — tente um áudio mais limpo antes de trocar de ferramenta.
2. **Soa humano?** Respiração, variação de ritmo. Voz clonada costuma ficar plana demais em frase longa.
3. **O sotaque incomoda?** Vai soar como ela falando inglês com sotaque brasileiro — o clone preserva o timbre, não troca o sotaque. Pra UGC isso costuma ser bom; pra locução institucional, não.

A duração vai mudar: inglês costuma sair 10–20% mais curto que português. Se for usar no gerador de criativos, monte com o áudio novo em vez de encaixar no projeto antigo.

## Se não prestar

Antes de desistir, tente nesta ordem:

1. **Áudio de referência melhor** — gravação limpa, sem música, 30s ou mais.
2. **Texto mais curto por pedaço** — baixe o teto de 220 para 150 caracteres.
3. **Outro motor** — OpenVoice v2 (MIT, serve pra produção) ou StyleTTS2.

Se mesmo assim não convencer, é sinal de que o caminho pago resolve e este não.